# Fashion Trend Prediction
**What is in this season? Using machine learning to identify fashion trends**

Supervisor: Dr Ollie Bartlett

### Research Questions
- **RQ1**: Can ML predict next season's best-selling products?
- **RQ2**: Which features most influence fashion trends?
- **RQ3**: Which ML algorithm performs best?
- **RQ4**: Can deep learning outperform traditional ML?
- **RQ5**: Can external data improve predictions?

In [ ]:
# 1. IMPORTS
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import seaborn as sns

import fashion_trend_utils as ftu


In [ ]:
# 2. LOAD, PREPROCESS AND MERGE DATA
data = ftu.load_and_prepare_data()


In [ ]:
# 3. AGGREGATE BY CATEGORY + SEASON + YEAR, THEN ADD LAG FEATURES
agg_data = ftu.aggregate_by_season(data)
print(f'Aggregated: {agg_data.shape}')

agg_data = ftu.add_lag_features(agg_data)
print(f'After lag features: {agg_data.shape}')


In [ ]:
# 4. ENCODE CATEGORIES AND BUILD THE FEATURE MATRIX
ftu.encode_categoricals(agg_data, verbose=True)

X, y, feature_cols, numeric_feats = ftu.build_features(agg_data)
print(f'Features: {len(feature_cols)}, Samples: {len(X)}')


In [ ]:
# 5. TIME-BASED TRAIN/TEST SPLIT AND SCALING
X_train, X_test, y_train, y_test, agg_data_sorted, split_idx = ftu.time_based_split(agg_data, X, y)
X_train, X_test, scaler = ftu.scale_features(X_train, X_test, numeric_feats)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')


In [ ]:
# 6. TRAIN MODELS
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, depth=10, learning_rate=0.1, random_state=42, verbose=0)
}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)

results = ftu.evaluate_models(models, X_test, y_test)


In [ ]:
# 7. RESULTS COMPARISON
results_df = pd.DataFrame(results).T
results_df


In [ ]:
# 8. BEST PREDICTIONS
best_model = models[ftu.best_model_name(results)]
pred_df = ftu.save_predictions(
    agg_data_sorted, split_idx, best_model.predict(X_test), y_test,
    'next_season_predictions.csv'
)
pred_df


In [ ]:
# 9. FEATURE IMPORTANCE
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': best_model.feature_importances_})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, y='feature', x='importance', palette='viridis')
plt.title('Top 15 Features (best model)')
plt.tight_layout()
plt.show()


In [ ]:
# 10. SEASONAL TRENDS VISUALIZATION
seasonal = agg_data.groupby(['Year', 'Season'])['total_quantity'].sum().reset_index()
plt.figure(figsize=(12, 5))
sns.lineplot(data=seasonal, x='Year', y='total_quantity', hue='Season', marker='o')
plt.title('Sales by Season')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 11. COLOR POPULARITY BY SEASON
color_s = agg_data.groupby(['Season', 'ProductColor'])['total_quantity'].sum().reset_index()
top_colors = color_s.sort_values('total_quantity', ascending=False).groupby('Season', group_keys=False).head(5)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_colors, x='Season', y='total_quantity', hue='ProductColor', palette='Set2')
plt.title('Top 5 Colors by Season')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## Summary
- **Best model**: XGBoost (R² = ?)
- **Top features**: city_count, revenue/quantity rolling averages
- **Next season prediction**: Run cell 8 to see top predicted best-sellers

To improve further: add weather, social media, or holiday data by joining on Date/Season/Year.